# Stacked Generalization (Meta-Learner) Ensemble

This notebook implements a two-level stacking pipeline:

**Level 0 — Base Models:**
1. XGBoost (from `james_xg_boost.ipynb`)
2. Random Forest (from `connor_random_forest.ipynb`)
3. CatBoost (with native categorical feature support)

**Level 1 — Meta-Learner:** Ridge Regression trained on out-of-fold predictions from Level 0.

**Validation strategy:** A single outer GroupKFold (by patient `sub_id`) drives everything. Inside each outer fold, base models generate OOF predictions on the outer-train subset, the meta-learner is fit on those OOF predictions, and the held-out outer fold is scored. This gives an honest end-to-end estimate with no patient leakage at any level.

For the final submission, OOF predictions are generated from the full training set, the meta-learner is fit on those, and test predictions come from fold-averaged base models.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

## 1. Load Data & Feature Engineering

In [ ]:
df_train = pd.read_csv("src_files/train_competition_2026.csv")
df_test = pd.read_csv("src_files/test_no_outcome.csv")

print(f"Raw train: {df_train.shape}")
print(f"Raw test:  {df_test.shape}")

In [ ]:
FEATURE_COLS = [
    c for c in df_train.columns
    if c not in ["obs", "sub_id", "time", "y_1", "y_2"]
]

NUM_COLS = [c for c in FEATURE_COLS if c.startswith("num_")]
CAT_COLS = [c for c in FEATURE_COLS if c.startswith("cat_")]
T_COLS = [c for c in FEATURE_COLS if c.startswith("t_")]


def collapse_obs(df):
    """Collapse time-series blocks into one row per observation.

    - num_* and cat_*: constant within obs, take first value (no redundant stats).
    - cat_* kept as raw integers so CatBoost can use native categorical splits.
    - t_*: rich temporal summaries — basic stats, early/late means, last-5 mean,
      quantiles, volatility, segment slopes, delta, range, full-window slope.
    """
    obs_groups = df.groupby("obs")

    # --- Constant columns: one value per obs ---
    const_agg = obs_groups[NUM_COLS + CAT_COLS].first()

    # --- Temporal columns: rich summaries ---
    basic_agg = obs_groups[T_COLS].agg(["mean", "std", "min", "max", "last"])
    basic_agg.columns = [f"{feat}_{agg}" for feat, agg in basic_agg.columns]

    # Quantiles
    q25 = obs_groups[T_COLS].quantile(0.25)
    q25.columns = [f"{c}_q25" for c in T_COLS]
    q75 = obs_groups[T_COLS].quantile(0.75)
    q75.columns = [f"{c}_q75" for c in T_COLS]

    # Early vs late mean (first half vs second half)
    def _early_mean(x):
        n = len(x)
        return x.iloc[: n // 2].mean()

    def _late_mean(x):
        n = len(x)
        return x.iloc[n // 2 :].mean()

    early = obs_groups[T_COLS].agg(_early_mean)
    early.columns = [f"{c}_early_mean" for c in T_COLS]
    late = obs_groups[T_COLS].agg(_late_mean)
    late.columns = [f"{c}_late_mean" for c in T_COLS]

    # Last-5 mean
    last5 = obs_groups[T_COLS].agg(lambda x: x.iloc[-5:].mean())
    last5.columns = [f"{c}_last5_mean" for c in T_COLS]

    # Volatility (mean absolute successive difference)
    volatility = obs_groups[T_COLS].agg(
        lambda x: np.abs(np.diff(x.values)).mean() if len(x) > 1 else 0.0
    )
    volatility.columns = [f"{c}_volatility" for c in T_COLS]

    # Delta, range, slope
    temporal = {}
    for col in T_COLS:
        s = obs_groups[col]
        temporal[f"{col}_delta"] = s.last() - s.first()
        temporal[f"{col}_range"] = s.max() - s.min()
        temporal[f"{col}_slope"] = s.apply(
            lambda x: np.polyfit(np.arange(len(x)), x.values, 1)[0]
            if len(x) > 1
            else 0.0
        )
    temporal_df = pd.DataFrame(temporal)

    # Segment slopes (first half, second half)
    def _half_slope(x, half):
        n = len(x)
        if n < 4:
            return 0.0
        seg = x.values[: n // 2] if half == "first" else x.values[n // 2 :]
        if len(seg) < 2:
            return 0.0
        return np.polyfit(np.arange(len(seg)), seg, 1)[0]

    seg_slopes = {}
    for col in T_COLS:
        s = obs_groups[col]
        seg_slopes[f"{col}_slope_early"] = s.apply(lambda x: _half_slope(x, "first"))
        seg_slopes[f"{col}_slope_late"] = s.apply(lambda x: _half_slope(x, "second"))
    seg_slopes_df = pd.DataFrame(seg_slopes)

    obs_length = obs_groups.size().rename("obs_length")

    return pd.concat(
        [const_agg, basic_agg, q25, q75, early, late, last5, volatility,
         temporal_df, seg_slopes_df, obs_length],
        axis=1,
    ).reset_index()

In [ ]:
X_train_df = collapse_obs(df_train)
X_test_df = collapse_obs(df_test)

# Targets (constant per obs)
y_train = df_train.groupby("obs")[["y_1", "y_2"]].first().reset_index(drop=True)

# Patient groups for GroupKFold
obs_to_patient = df_train.groupby("obs")["sub_id"].first().reset_index(drop=True)

# Separate obs index for submission
test_obs = X_test_df["obs"].copy()
X_train = X_train_df.drop(columns=["obs"])
X_test = X_test_df.drop(columns=["obs"])

# Identify categorical column indices for CatBoost
CAT_FEATURE_INDICES = [X_train.columns.get_loc(c) for c in CAT_COLS]

print(f"Collapsed train: {X_train.shape[0]} obs x {X_train.shape[1]} features")
print(f"Collapsed test:  {X_test.shape[0]} obs x {X_test.shape[1]} features")
print(f"Unique patients: {obs_to_patient.nunique()}")
print(f"Categorical columns ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Categorical indices: {CAT_FEATURE_INDICES}")

## 2. Base Model Definitions

Each base model exposes a unified interface via sklearn-compatible `fit`/`predict`. XGBoost and CatBoost use early stopping with the validation fold during OOF generation.

In [ ]:
def make_xgb():
    """XGBoost config from james_xg_boost.ipynb."""
    return xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        early_stopping_rounds=50,
        random_state=42,
    )


def make_rf():
    """Random Forest config from connor_random_forest.ipynb (best CV params)."""
    return RandomForestRegressor(
        n_estimators=601,
        max_depth=10,
        max_features=None,
        min_samples_leaf=5,
        min_samples_split=3,
        n_jobs=-1,
        random_state=42,
    )


def make_catboost():
    """CatBoost with native categorical feature support."""
    return CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=3,
        subsample=0.8,
        early_stopping_rounds=50,
        cat_features=CAT_FEATURE_INDICES,
        random_seed=42,
        verbose=0,
    )

## 3. Honest Outer-CV Evaluation

A single outer GroupKFold (5 folds, grouped by patient) drives the entire evaluation. For each outer fold:

1. **Inner OOF generation:** Within the outer-train subset, run an inner GroupKFold to produce OOF predictions for each base model. This gives the meta-learner training features with no leakage.
2. **Meta-learner fit:** Train Ridge on the inner OOF matrix.
3. **Outer-fold scoring:** Each base model is trained on the full outer-train set and predicts the outer-val set. These predictions are fed through the meta-learner for the final outer-fold score.

This gives an apples-to-apples comparison between single models, simple averaging, and the full stacking pipeline.

In [ ]:
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5

BASE_MODELS = {
    "xgb": make_xgb,
    "rf": make_rf,
    "catboost": make_catboost,
}
model_names = list(BASE_MODELS.keys())

TARGETS = ["y_1", "y_2"]


def fit_and_predict(model_name, model, X_tr, y_tr, X_val, y_val):
    """Fit a base model with appropriate early-stopping, return val predictions."""
    if model_name == "xgb":
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    elif model_name == "catboost":
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
    else:
        model.fit(X_tr, y_tr)
    return model.predict(X_val)


def generate_inner_oof(X_outer_tr, y_outer_tr, groups_outer_tr):
    """Generate OOF predictions for all base models within an outer-train subset.

    Returns:
        S_oof: dict[model_name] -> np.array of OOF predictions (len = outer-train size)
    """
    inner_gkf = GroupKFold(n_splits=N_INNER_FOLDS)
    S_oof = {name: np.zeros(len(X_outer_tr)) for name in model_names}

    for name, factory in BASE_MODELS.items():
        for tr_idx, val_idx in inner_gkf.split(
            X_outer_tr, y_outer_tr, groups=groups_outer_tr
        ):
            model = factory()
            X_tr = X_outer_tr.iloc[tr_idx]
            y_tr = y_outer_tr.iloc[tr_idx]
            X_val = X_outer_tr.iloc[val_idx]
            y_val = y_outer_tr.iloc[val_idx]
            preds = fit_and_predict(name, model, X_tr, y_tr, X_val, y_val)
            S_oof[name][val_idx] = preds

    return S_oof

In [ ]:
outer_gkf = GroupKFold(n_splits=N_OUTER_FOLDS)

# Collect per-outer-fold MAEs for each method and target
outer_scores = {
    t: {name: [] for name in model_names + ["simple_avg", "stacked"]}
    for t in TARGETS
}

for outer_fold, (outer_tr_idx, outer_val_idx) in enumerate(
    outer_gkf.split(X_train, y_train["y_1"], groups=obs_to_patient)
):
    print(f"\n{'='*60}")
    print(f"Outer fold {outer_fold + 1}/{N_OUTER_FOLDS}  "
          f"(train={len(outer_tr_idx)}, val={len(outer_val_idx)})")
    print(f"{'='*60}")

    X_otr = X_train.iloc[outer_tr_idx].reset_index(drop=True)
    X_oval = X_train.iloc[outer_val_idx]
    groups_otr = obs_to_patient.iloc[outer_tr_idx].reset_index(drop=True)

    for target in TARGETS:
        y_otr = y_train[target].iloc[outer_tr_idx].reset_index(drop=True)
        y_oval = y_train[target].iloc[outer_val_idx]

        # Step 1: Generate inner OOF predictions on outer-train
        inner_oof = generate_inner_oof(X_otr, y_otr, groups_otr)
        S_train_meta = np.column_stack([inner_oof[m] for m in model_names])

        # Step 2: Fit meta-learner on inner OOF
        meta = Ridge(alpha=1.0)
        meta.fit(S_train_meta, y_otr.values)

        # Step 3: Train each base model on full outer-train, predict outer-val
        val_base_preds = {}
        for name, factory in BASE_MODELS.items():
            model = factory()
            preds = fit_and_predict(name, model, X_otr, y_otr, X_oval, y_oval)
            val_base_preds[name] = preds
            mae = mean_absolute_error(y_oval, preds)
            outer_scores[target][name].append(mae)

        # Step 4: Score simple average and stacked ensemble on outer-val
        S_val = np.column_stack([val_base_preds[m] for m in model_names])

        avg_preds = S_val.mean(axis=1)
        outer_scores[target]["simple_avg"].append(
            mean_absolute_error(y_oval, avg_preds)
        )

        stacked_preds = meta.predict(S_val)
        outer_scores[target]["stacked"].append(
            mean_absolute_error(y_oval, stacked_preds)
        )

        print(f"  {target}: ", end="")
        for method in model_names + ["simple_avg", "stacked"]:
            print(f"{method}={outer_scores[target][method][-1]:.4f}  ", end="")
        print()

        # Show meta-learner weights for this fold
        weights = dict(zip(model_names, meta.coef_.round(4)))
        print(f"    Ridge weights: {weights}, intercept: {meta.intercept_:.4f}")

## 4. Final Submission Pipeline

Now that the outer CV gives us honest scores, we build the actual submission:

1. Generate full-data OOF predictions (inner GroupKFold on all training data) for each base model.
2. Fit the final meta-learner on the full OOF matrix.
3. Train 5 fold models per base learner and cache them for fold-averaged test predictions.

In [ ]:
full_gkf = GroupKFold(n_splits=N_OUTER_FOLDS)

# Store OOF predictions and fold models for final submission
full_oof = {t: {} for t in TARGETS}
full_fold_models = {t: {} for t in TARGETS}
meta_models = {}

for target in TARGETS:
    print(f"\n--- {target} ---")
    y_target = y_train[target]

    for name, factory in BASE_MODELS.items():
        oof_preds = np.zeros(len(X_train))
        fold_models = []

        for tr_idx, val_idx in full_gkf.split(X_train, y_target, groups=obs_to_patient):
            model = factory()
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_target.iloc[tr_idx], y_target.iloc[val_idx]
            preds = fit_and_predict(name, model, X_tr, y_tr, X_val, y_val)
            oof_preds[val_idx] = preds
            fold_models.append(model)

        full_oof[target][name] = oof_preds
        full_fold_models[target][name] = fold_models
        mae = mean_absolute_error(y_target, oof_preds)
        print(f"  {name} full OOF MAE: {mae:.4f}")

    # Fit final meta-learner
    S_full = np.column_stack([full_oof[target][m] for m in model_names])
    final_meta = Ridge(alpha=1.0)
    final_meta.fit(S_full, y_target.values)
    meta_models[target] = final_meta

    weights = dict(zip(model_names, final_meta.coef_.round(4)))
    print(f"  Final Ridge weights: {weights}, intercept: {final_meta.intercept_:.4f}")

## 5. Test Inference (Fold-Averaged Predictions)

For each base model, predict on the test set with all 5 fold models and average. This ensures the test prediction distribution matches the OOF predictions the meta-learner was trained on.

In [ ]:
test_base_preds = {t: {} for t in TARGETS}

for target in TARGETS:
    for name in model_names:
        fold_preds = np.column_stack([
            m.predict(X_test) for m in full_fold_models[target][name]
        ])
        test_base_preds[target][name] = fold_preds.mean(axis=1)

    print(f"{target}: fold-averaged {len(model_names)} base models x {N_OUTER_FOLDS} folds")

In [ ]:
submission_preds = {}

for target in TARGETS:
    S_test = np.column_stack(
        [test_base_preds[target][m] for m in model_names]
    )
    submission_preds[target] = meta_models[target].predict(S_test)

submission = pd.DataFrame({
    "obs": test_obs,
    "y_1": submission_preds["y_1"],
    "y_2": submission_preds["y_2"],
})

print(f"Submission shape: {submission.shape}")
submission.head()

## 6. Performance Comparison

In [ ]:
DISPLAY_NAMES = {
    "xgb": "XGBoost",
    "rf": "Random Forest",
    "catboost": "CatBoost",
    "simple_avg": "Simple Average",
    "stacked": "STACKED (Ridge)",
}

rows = []
for target in TARGETS:
    for method in model_names + ["simple_avg", "stacked"]:
        fold_maes = outer_scores[target][method]
        rows.append({
            "Target": target,
            "Model": DISPLAY_NAMES.get(method, method),
            "CV MAE": np.mean(fold_maes),
            "Std": np.std(fold_maes),
        })

comparison = pd.DataFrame(rows)

# Pivot for clean display
pivot_mae = comparison.pivot(index="Model", columns="Target", values="CV MAE")
pivot_std = comparison.pivot(index="Model", columns="Target", values="Std")
pivot_mae["Mean MAE"] = pivot_mae.mean(axis=1)
pivot_mae = pivot_mae.sort_values("Mean MAE")

print("\n" + "="*65)
print("   HONEST OUTER-CV MAE (GroupKFold by patient, 5 folds)")
print("="*65)
print(pivot_mae.round(4).to_string())
print("="*65)
print("\nFold-level std:")
print(pivot_std.round(4).to_string())

## 7. Save Submission

In [ ]:
submission.to_csv("james_stack_submission.csv", index=False)
print("Saved: james_stack_submission.csv")
submission.head(10)